<a href="https://colab.research.google.com/github/L0ckR/VK_RECSYS_COURSE/blob/main/vk_recsys_course_HW1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task
Examine the provided `solution_toppop.py` script, then load and preprocess `train.parquet` and `test.parquet` by mapping `user_id` and `item_id` to contiguous integers and creating an implicit feedback matrix. Train an Alternating Least Squares (ALS) collaborative filtering model using the `implicit` library on the preprocessed training data. Generate top 10 item recommendations for each unique user in `test.parquet`, ensuring that already interacted items are filtered out. Finally, create a complete Python script that encompasses all these steps and saves the recommendations to a CSV file in the format `user_id`, `recs`.

## Examine Baseline

### Subtask:
Review the provided `solution_toppop.py` script to understand the existing popularity-based baseline. This will help in setting a performance target and understanding the expected output format.


**Reasoning**:
To review the `solution_toppop.py` script as instructed, I need to display its content. Using `cat` command in a code block will achieve this.



In [5]:
cat solution_toppop.py

import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity
import argparse

def main(input_path: str, output_path: str):
    train = pd.read_parquet(input_path)

    top = (train
           .groupby("item_id", as_index=False)["user_id"]
           .count()
           .sort_values("user_id", ascending=False)
           )
    top10 = top[:10]["item_id"].to_list()

    users_for_rec = pd.DataFrame({"user_id": train["user_id"].unique()})

    print("Compute recommendations. It may take a few minutes.")
    users_for_rec['recs'] = [top10 for _ in range(len(users_for_rec))]
    print("Recommendations computed.")

    result = users_for_rec.explode("recs")

    print(f"Save result to file {output_path}.")
    result.to_csv(output_path, index=False)
    print("File saved.")

if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="Recommender arguments.")
    parser.add_argument("--input_path", type=str,

```markdown
### Review of `solution_toppop.py`

The `solution_toppop.py` script implements a popularity-based recommendation system. Here's a breakdown of its functionality:

1.  **Data Loading**: The script loads the training data from a parquet file specified by `input_path` into a pandas DataFrame using `pd.read_parquet(input_path)`.

2.  **Popularity Calculation**: It calculates item popularity by grouping the training data by `item_id` and counting the number of `user_id` interactions for each item. This count represents the popularity score. The items are then sorted in descending order of popularity.

    ```python
    top = (train
           .groupby("item_id", as_index=False)["user_id"]
           .count()
           .sort_values("user_id", ascending=False)
           )
    ```

3.  **Recommendation Generation**: The script selects the top 10 most popular `item_id`s from the sorted list. These top 10 items constitute the recommendation list for *all* users.

    ```python
    top10 = top[:10]["item_id"].to_list()
    ```

    It then identifies all unique `user_id`s present in the training data. For each unique user, it assigns the same list of top 10 popular items as recommendations.

    ```python
    users_for_rec = pd.DataFrame({"user_id": train["user_id"].unique()})
    users_for_rec['recs'] = [top10 for _ in range(len(users_for_rec))]
    ```

4.  **Key Data Structures and Methods**: The primary data structure used is a pandas DataFrame. Key methods include `pd.read_parquet()`, `groupby()`, `count()`, `sort_values()`, `unique()`, and `explode()`. The `explode()` method is used to transform each element of a list-like entry (the `recs` column) into a separate row, replicating the index values.

5.  **Output Format**: The final recommendations are saved to a CSV file (specified by `output_path`). The output DataFrame, after exploding the recommendations, will have two columns:
    *   `user_id`: The ID of the user.
    *   `recs`: An `item_id` that is recommended to the `user_id`. Each `user_id` will have 10 rows, corresponding to the 10 recommended `item_id`s.

    ```python
    result.to_csv(output_path, index=False)
    ```
    The CSV file will contain `user_id` and `recs` (which is an `item_id`) columns, without an index. The data types will likely be integers for both columns, reflecting the original `user_id` and `item_id` types.
```

## Load and Preprocess Data

### Subtask:
Load the `train.parquet` and `test.parquet` files into pandas DataFrames. Map `user_id` and `item_id` to contiguous integer IDs. Create an implicit feedback matrix (e.g., using `timespent` as interaction strength) from the training data, suitable for collaborative filtering models.


**Reasoning**:
The first step is to load the necessary dataframes and display their initial rows to understand their structure and verify successful loading. This aligns with instructions 1 in the subtask.



In [6]:
import pandas as pd

train_df = pd.read_parquet('/content/train.parquet')
test_df = pd.read_parquet('/content/test.parquet')

print("Train DataFrame Head:")
print(train_df.head())
print("\nTest DataFrame Head:")
print(test_df.head())

Train DataFrame Head:
     user_id    item_id  place  platform  agent  timespent   like  dislike  \
0  141827770  160593626      1         1      1         56  False    False   
1  494341617  402842289      1         0      0         11  False    False   
2  453833313  102672169      0         0      0         45  False    False   
3  154047442  139795075      1         0      0         39  False    False   
4  330326695  232077218      1         0      0         57  False    False   

   share  bookmark  click_on_author  open_comments  
0  False     False            False          False  
1  False     False            False          False  
2  False     False            False          False  
3  False     False            False          False  
4  False     False            False          False  

Test DataFrame Head:
     user_id    item_id  place  platform  agent  timespent   like  dislike  \
0  506947605  321354762      1         0      0          2  False    False   
1  168252888 

**Reasoning**:
To create a comprehensive mapping for both user_id and item_id, it's essential to collect all unique IDs from both the training and testing datasets. This ensures that all IDs across both datasets will have a consistent mapping to contiguous integers, preventing issues when processing either dataset individually or when combining them later. This step addresses instructions 2, 3, 4, and 5.



In [7]:
all_user_ids = pd.concat([train_df['user_id'], test_df['user_id']]).unique()
all_item_ids = pd.concat([train_df['item_id'], test_df['item_id']]).unique()

user_id_to_idx = {user_id: idx for idx, user_id in enumerate(all_user_ids)}
item_id_to_idx = {item_id: idx for idx, item_id in enumerate(all_item_ids)}

print(f"Number of unique users: {len(all_user_ids)}")
print(f"Number of unique items: {len(all_item_ids)}")
print("User ID to Index Mapping (first 5):", dict(list(user_id_to_idx.items())[:5]))
print("Item ID to Index Mapping (first 5):", dict(list(item_id_to_idx.items())[:5]))

Number of unique users: 8531
Number of unique items: 6652
User ID to Index Mapping (first 5): {np.int64(141827770): 0, np.int64(494341617): 1, np.int64(453833313): 2, np.int64(154047442): 3, np.int64(330326695): 4}
Item ID to Index Mapping (first 5): {np.int64(160593626): 0, np.int64(402842289): 1, np.int64(102672169): 2, np.int64(139795075): 3, np.int64(232077218): 4}


**Reasoning**:
Now that the mappings from original `user_id` and `item_id` to contiguous integer indices have been created, I will apply these mappings to both `train_df` and `test_df` to add `user_idx` and `item_idx` columns. This directly addresses instructions 6 and 7 of the subtask.



In [8]:
train_df['user_idx'] = train_df['user_id'].map(user_id_to_idx)
train_df['item_idx'] = train_df['item_id'].map(item_id_to_idx)
test_df['user_idx'] = test_df['user_id'].map(user_id_to_idx)
test_df['item_idx'] = test_df['item_id'].map(item_id_to_idx)

print("Train DataFrame with new indices (head):")
print(train_df[['user_id', 'item_id', 'user_idx', 'item_idx', 'timespent']].head())
print("\nTest DataFrame with new indices (head):")
print(test_df[['user_id', 'item_id', 'user_idx', 'item_idx', 'timespent']].head())

Train DataFrame with new indices (head):
     user_id    item_id  user_idx  item_idx  timespent
0  141827770  160593626         0         0         56
1  494341617  402842289         1         1         11
2  453833313  102672169         2         2         45
3  154047442  139795075         3         3         39
4  330326695  232077218         4         4         57

Test DataFrame with new indices (head):
     user_id    item_id  user_idx  item_idx  timespent
0  506947605  321354762      1986      1596          2
1  168252888  247953018      5130       722         10
2  437389696  538636793      4254      3620          3
3  286667180  105081016      3522      3518          1
4   76986715  297168940       690      3613         11


**Reasoning**:
To construct the implicit feedback matrix, I will use `scipy.sparse.csr_matrix`. The matrix will be built from the `train_df` using the `user_idx` as row indices, `item_idx` as column indices, and `timespent` as the data values. The dimensions of the matrix will be based on the total number of unique users and items found across both training and test sets, as determined by `all_user_ids` and `all_item_ids`. This directly addresses instruction 8 of the subtask.



In [9]:
from scipy.sparse import csr_matrix

num_users = len(all_user_ids)
num_items = len(all_item_ids)

# Create the implicit feedback matrix from train_df
# The values in the matrix will be 'timespent' as interaction strength
# Rows are user_idx, columns are item_idx
user_item_matrix = csr_matrix((
    train_df['timespent'].astype(float),
    (train_df['user_idx'], train_df['item_idx'])
), shape=(num_users, num_items))

print(f"Shape of the implicit feedback matrix: {user_item_matrix.shape}")
print(f"Number of non-zero elements in the matrix: {user_item_matrix.nnz}")
print("Implicit feedback matrix created successfully.")

Shape of the implicit feedback matrix: (8531, 6652)
Number of non-zero elements in the matrix: 1904581
Implicit feedback matrix created successfully.


## Train Collaborative Filtering Model

### Subtask:
Train an Alternating Least Squares (ALS) model from the `implicit` library using the preprocessed implicit feedback matrix from the training data. This step will involve selecting appropriate hyperparameters for the model.


**Reasoning**:
I need to install the `implicit` library first, as it's not a standard library in Colab environments, to ensure the ALS model can be imported and used without errors.



In [ ]:
!uv pip install implicit

Using Python 3.12.12 environment at: /usr
Resolved 5 packages in 193ms
⠋ Preparing packages... (0/1)                                                   